# **CRISP-DM Case Study: Airline Passenger Satisfaction Prediction**

### **Required Libraries**

In [10]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

In [11]:
import os
import pickle

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

### **Formattings**

In [12]:
# ---------- Global plotting style ----------
sns.set_style("whitegrid")
PALETTE = ["#1F4E79", "#2E86AB", "#5DA9C0", "#A23B72", "#F18F01", "#3B9C6D"]
ACCENT = "#1F4E79"
ACCENT2 = "#C0392B"
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "font.size": 10,
})

### **Functions**

In [ ]:
def annotate_bars(ax, fmt="{:.0f}", fontsize=9, color="black", offset=0.01, pct_total=None):
    for p in ax.patches:
        height = p.get_height()
        width = p.get_width()
        # Vertical bars
        if height != 0 and abs(height) >= abs(width):
            x = p.get_x() + width / 2
            y = height
            label = fmt.format(height)
            if pct_total:
                label += f"\n({height / pct_total * 100:.1f}%)"
            ax.annotate(label, (x, y), ha="center", va="bottom",
                        fontsize=fontsize, color=color,
                        xytext=(0, 3), textcoords="offset points")
        else:
            y = p.get_y() + height / 2
            x = width
            label = fmt.format(width)
            ax.annotate(label, (x, y), ha="left", va="center",
                        fontsize=fontsize, color=color,
                        xytext=(3, 0), textcoords="offset points")

## **PHASE 2: DATA UNDERSTANDING**

In [14]:
train_raw = pd.read_csv("data/train.csv")
test_raw = pd.read_csv("data/test.csv")

In [15]:
train_raw.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [25]:
train_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Gender                             103904 non-null  object 
 1   Customer Type                      103904 non-null  object 
 2   Age                                103904 non-null  int64  
 3   Type of Travel                     103904 non-null  object 
 4   Class                              103904 non-null  object 
 5   Flight Distance                    103904 non-null  int64  
 6   Inflight wifi service              103904 non-null  int64  
 7   Departure/Arrival time convenient  103904 non-null  int64  
 8   Ease of Online booking             103904 non-null  int64  
 9   Gate location                      103904 non-null  int64  
 10  Food and drink                     103904 non-null  int64  
 11  Online boarding                    1039

In [16]:
test_raw.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,19556,Female,Loyal Customer,52,Business travel,Eco,160,5,4,...,5,5,5,5,2,5,5,50,44.0,satisfied
1,1,90035,Female,Loyal Customer,36,Business travel,Business,2863,1,1,...,4,4,4,4,3,4,5,0,0.0,satisfied
2,2,12360,Male,disloyal Customer,20,Business travel,Eco,192,2,0,...,2,4,1,3,2,2,2,0,0.0,neutral or dissatisfied
3,3,77959,Male,Loyal Customer,44,Business travel,Business,3377,0,0,...,1,1,1,1,3,1,4,0,6.0,satisfied
4,4,36875,Female,Loyal Customer,49,Business travel,Eco,1182,2,3,...,2,2,2,2,4,2,4,0,20.0,satisfied


In [26]:
test_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25976 entries, 0 to 25975
Data columns (total 23 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Gender                             25976 non-null  object 
 1   Customer Type                      25976 non-null  object 
 2   Age                                25976 non-null  int64  
 3   Type of Travel                     25976 non-null  object 
 4   Class                              25976 non-null  object 
 5   Flight Distance                    25976 non-null  int64  
 6   Inflight wifi service              25976 non-null  int64  
 7   Departure/Arrival time convenient  25976 non-null  int64  
 8   Ease of Online booking             25976 non-null  int64  
 9   Gate location                      25976 non-null  int64  
 10  Food and drink                     25976 non-null  int64  
 11  Online boarding                    25976 non-null  int

In [17]:
print(f"Train shape: {train_raw.shape}")
print(f"Test shape : {test_raw.shape}")

Train shape: (103904, 25)
Test shape : (25976, 25)


In [23]:
for df in (train_raw, test_raw):
    df.drop(columns=[c for c in ["Unnamed: 0", "id"] if c in df.columns],
            inplace=True, errors="ignore")
    print(f"After dropping columns, shape : {df.shape}")

After dropping columns, shape : (103904, 23)
After dropping columns, shape : (25976, 23)


In [ ]:
print("Column dtypes:")
print(train_raw.dtypes.value_counts())


Column dtypes:
int64      17
object      5
float64     1
Name: count, dtype: int64


In [29]:
print("Missing values (train dataset):")
missing = train_raw.isnull().sum()
print(missing[missing > 0])

Missing values (train dataset):
Arrival Delay in Minutes    310
dtype: int64


In [30]:
train_raw["Arrival Delay in Minutes"].fillna(train_raw["Arrival Delay in Minutes"].median(), inplace=True)

In [31]:
print("Missing values (train dataset):")
missing = train_raw.isnull().sum()
print(missing[missing > 0])

Missing values (train dataset):
Series([], dtype: int64)


In [32]:
print("Target class balance (train):")
print(train_raw["satisfaction"].value_counts(normalize=True).round(3))

Target class balance (train):
satisfaction
neutral or dissatisfied    0.567
satisfied                  0.433
Name: proportion, dtype: float64


In [ ]:
counts = train_raw["satisfaction"].value_counts()
total = counts.sum()
fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(counts.index, counts.values, color=[ACCENT, "#94B8D8"], width=0.55,
              edgecolor="white", linewidth=1.5)

for bar, val in zip(bars, counts.values):
    ax.annotate(f"{val:,}\n({val/total*100:.1f}%)",
                (bar.get_x() + bar.get_width() / 2, val),
                ha="center", va="bottom", fontsize=11, fontweight="bold",
                xytext=(0, 5), textcoords="offset points")
    
ax.set_title("Target Class Distribution (Train Set, n={:,})".format(total))
ax.set_ylabel("Number of Passengers")
ax.set_ylim(0, counts.max() * 1.18)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()
plt.savefig(f"{OUT_DIR}/01_target_distribution.png", dpi=150)
plt.close()

In [38]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, title in zip(
    axes, ["Class", "Type of Travel"],
    ["Satisfaction Rate by Travel Class", "Satisfaction Rate by Type of Travel"]
):
    ct = pd.crosstab(train_raw[col], train_raw["satisfaction"], normalize="index") * 100
    ct = ct[["satisfied", "neutral or dissatisfied"]]
    ct.plot(kind="bar", stacked=True, ax=ax, color=[ACCENT, "#D8D8D8"],
            edgecolor="white", width=0.6, legend=False)
    
    for i, (idx, row) in enumerate(ct.iterrows()):
        cum = 0
        for val in row:
            if val > 4:
                ax.text(i, cum + val / 2, f"{val:.1f}%", ha="center", va="center",
                        fontsize=10, fontweight="bold",
                        color="white" if cum < ct["satisfied"].max() else "black")
            cum += val
    ax.set_title(title)
    ax.set_ylabel("Share of Passengers (%)")
    ax.set_xlabel("")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=0)

handles = [plt.Rectangle((0, 0), 1, 1, color=ACCENT),
           plt.Rectangle((0, 0), 1, 1, color="#D8D8D8")]
fig.legend(handles, ["Satisfied", "Neutral / Dissatisfied"],
           loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.0), fontsize=10, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()
plt.savefig(f"{OUT_DIR}/02_satisfaction_by_class_travel.png", dpi=150)
plt.close()

In [39]:
numeric_cols = train_raw.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(12.5, 10))
corr = train_raw[numeric_cols].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f",
            annot_kws={"size": 6.5}, square=True, linewidths=0.3,
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})
plt.title("Correlation Heatmap of Numeric Features (values = Pearson r)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
plt.savefig(f"{OUT_DIR}/03_correlation_heatmap.png", dpi=150)
plt.close()

In [40]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, col, color in zip(axes, ["Age", "Flight Distance"], [ACCENT, "#3B9C6D"]):
    sns.histplot(train_raw[col], bins=30, kde=True, ax=ax, color=color, alpha=0.75)
    mean_v, med_v = train_raw[col].mean(), train_raw[col].median()
    ax.axvline(mean_v, color=ACCENT2, linestyle="--", linewidth=1.5,
               label=f"Mean = {mean_v:.1f}")
    ax.axvline(med_v, color="black", linestyle=":", linewidth=1.5,
               label=f"Median = {med_v:.1f}")
    ax.set_title(f"{col} Distribution")
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/04_age_distance_distribution.png", dpi=150)
plt.close()

In [41]:
age_bins = [0, 18, 30, 45, 60, 100]
age_labels = ["<18", "18-29", "30-44", "45-59", "60+"]
train_raw["Age Group"] = pd.cut(train_raw["Age"], bins=age_bins, labels=age_labels, right=False)
age_ct = pd.crosstab(train_raw["Age Group"], train_raw["satisfaction"], normalize="index") * 100
age_counts = train_raw["Age Group"].value_counts().reindex(age_labels)

fig, ax = plt.subplots(figsize=(8.5, 5))
bars = ax.bar(age_labels, age_ct.reindex(age_labels)["satisfied"], color=ACCENT,
              edgecolor="white", width=0.6)
for bar, val, n in zip(bars, age_ct.reindex(age_labels)["satisfied"], age_counts):
    ax.annotate(f"{val:.1f}%\n(n={n:,})", (bar.get_x() + bar.get_width() / 2, val),
                ha="center", va="bottom", fontsize=9.5, fontweight="bold",
                xytext=(0, 4), textcoords="offset points")
ax.set_title("Satisfaction Rate by Age Group")
ax.set_ylabel("% Satisfied")
ax.set_xlabel("Age Group")
ax.set_ylim(0, max(age_ct["satisfied"]) * 1.25)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/08_satisfaction_by_age_group.png", dpi=150)
plt.close()

In [42]:
train_raw["Total Delay (raw)"] = (train_raw["Departure Delay in Minutes"] +
                                   train_raw["Arrival Delay in Minutes"].fillna(0))
delay_bins = [-1, 0, 15, 60, 1600]
delay_labels = ["No delay", "1-15 min", "16-60 min", ">60 min"]
train_raw["Delay Bucket"] = pd.cut(train_raw["Total Delay (raw)"], bins=delay_bins, labels=delay_labels)
delay_ct = pd.crosstab(train_raw["Delay Bucket"], train_raw["satisfaction"], normalize="index") * 100
delay_counts = train_raw["Delay Bucket"].value_counts().reindex(delay_labels)

fig, ax = plt.subplots(figsize=(8.5, 5))
bars = ax.bar(delay_labels, delay_ct.reindex(delay_labels)["satisfied"], color="#F18F01",
              edgecolor="white", width=0.6)
for bar, val, n in zip(bars, delay_ct.reindex(delay_labels)["satisfied"], delay_counts):
    ax.annotate(f"{val:.1f}%\n(n={n:,})", (bar.get_x() + bar.get_width() / 2, val),
                ha="center", va="bottom", fontsize=9.5, fontweight="bold",
                xytext=(0, 4), textcoords="offset points")
ax.set_title("Satisfaction Rate by Total Delay (Departure + Arrival)")
ax.set_ylabel("% Satisfied")
ax.set_xlabel("Total Delay Bucket")
ax.set_ylim(0, max(delay_ct["satisfied"]) * 1.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/09_satisfaction_by_delay.png", dpi=150)
plt.close()

In [43]:
train_raw.drop(columns=["Age Group", "Total Delay (raw)", "Delay Bucket"], inplace=True)